# 🔄 LangGraph State Reducers & Channel Communication

### Overview & Learning Objectives
In LangGraph, **State** is composed of independent communication channels. By default, channels operate in **overwrite mode**: each node update replaces the prior value.

However, when multiple nodes run in parallel (fan-out) or when accumulating history (messages), overwrite mode causes **concurrent write conflicts** (`InvalidUpdateError`).

**Reducers** solve this by defining a reconciliation function:
$$\text{New State} = \text{Reducer}(\text{Current State}, \text{Update})$$

In this lab, we explore:
1. Default overwriting state behavior.
2. Parallel node execution conflicts without reducers.
3. Standard reducers using `Annotated[list, add]`.
4. Writing resilient custom reducers that handle `None` values safely.
5. The built-in `add_messages` reducer and message pruning via `RemoveMessage`.

## 📐 Architectural Diagram: Overwrite vs Reducer Reconciliation

The comparison below illustrates why state reducers are essential for parallel execution:

<div align="center">
  <img src="images/07_state_reducers_comparison.png" alt="LangGraph State Reducers: Overwrite Conflict vs Reducer Reconciliation" width="100%" />
</div>

<br/>

<details>
<summary><b>🔍 View Raw Mermaid Diagram Syntax</b></summary>

```mermaid
flowchart TD
    subgraph Conflict [WITHOUT REDUCER: Concurrent Overwrite Conflict]
        direction TB
        P_START([START]) --> P_N1[Node 1: foo=1]
        P_N1 --> P_N2[Node 2: foo=2]
        P_N1 --> P_N3[Node 3: foo=3]
        P_N2 & P_N3 -->|Concurrent write to 'foo'| P_ERR[💥 InvalidUpdateError: At least two parallel updates]
    end

    subgraph Resolved [WITH REDUCER: Annotated foo: list, add]
        direction TB
        R_START([START]) --> R_N1[Node 1: foo=[1]]
        R_N1 --> R_N2[Node 2: foo=[2]]
        R_N1 --> R_N3[Node 3: foo=[3]]
        R_N2 & R_N3 -->|Channel Reducer: operator.add| R_REC[🔄 foo = [1, 2, 3]]
        R_REC --> R_END([END])
    end
```

</details>


## 1. Installation

Install LangGraph and core libraries.

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_core langgraph

## 2. Default Overwrite Behavior (Linear Graph)

In a linear graph without reducers, each node simply replaces the previous state key.

In [ ]:
from typing_extensions import TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    foo: int

def node_1(state):
    print("---Node 1---")
    return {"foo": state['foo'] + 1}

# Build graph
builder = StateGraph(State)
builder.add_node("node_1", node_1)

# Logic
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

Execute the graph: `node_1` receives `foo=1`, returns `foo=2`, overwriting the initial state.

In [ ]:
graph.invoke({"foo" : 1})

## 3. The Problem: Parallel Writes Cause `InvalidUpdateError`

When a graph fans out from `node_1` into two parallel nodes (`node_2` and `node_3`), both attempt to overwrite `foo` in the exact same superstep.

In [ ]:
class State(TypedDict):
    foo: int

def node_1(state):
    print("---Node 1---")
    return {"foo": state['foo'] + 1}

def node_2(state):
    print("---Node 2---")
    return {"foo": state['foo'] + 1}

def node_3(state):
    print("---Node 3---")
    return {"foo": state['foo'] + 1}

# Build graph
builder = StateGraph(State)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# Logic
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_1", "node_3")
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

Invoking this fan-out graph raises an `InvalidUpdateError` because LangGraph cannot determine which update should take precedence.

In [ ]:
from langgraph.errors import InvalidUpdateError
try:
    graph.invoke({"foo" : 1})
except InvalidUpdateError as e:
    print(f"InvalidUpdateError occurred: {e}")


## 4. Resolving Conflicts with Reducers (`operator.add`)

We attach a reducer function to the state key using Python's `Annotated` typing:
```python
from operator import add
from typing import Annotated

class State(TypedDict):
    foo: Annotated[list[int], add]
```
Now, when parallel nodes emit updates, LangGraph applies `add` to combine the lists rather than overwriting.

In [ ]:
from operator import add
from typing import Annotated

class State(TypedDict):
    foo: Annotated[list[int], add]

def node_1(state):
    print("---Node 1---")
    return {"foo": [state['foo'][0] + 1]}

# Build graph
builder = StateGraph(State)
builder.add_node("node_1", node_1)

# Logic
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

In linear execution, `node_1` appends `[2]` to initial `[1]`, yielding `[1, 2]`.

In [ ]:
graph.invoke({"foo" : [1]})

### Testing Reducers with Parallel Execution
We rebuild the fan-out graph (`node_1 -> [node_2, node_3] -> END`).

In [ ]:
def node_1(state):
    print("---Node 1---")
    return {"foo": [state['foo'][-1] + 1]}

def node_2(state):
    print("---Node 2---")
    return {"foo": [state['foo'][-1] + 1]}

def node_3(state):
    print("---Node 3---")
    return {"foo": [state['foo'][-1] + 1]}

# Build graph
builder = StateGraph(State)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# Logic
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_1", "node_3")
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

Now invoking the parallel graph succeeds! Both updates `[2]` and `[3]` are cleanly combined with `[1]` to produce `[1, 2, 3]`.

In [ ]:
graph.invoke({"foo" : [1]})

## 5. The `None` Pitfall with Standard `operator.add`

`operator.add` assumes both operands are lists. If an initial state passes `None`, `operator.add(None, [2])` raises a `TypeError`.

In [ ]:
try:
    graph.invoke({"foo" : None})
except TypeError as e:
    print(f"TypeError occurred: {e}")

## 6. Authoring Custom Reducers

A custom reducer function `reduce_list(left, right)` safely inspects both operands, initializing empty lists when `None` is encountered.

In [ ]:
def reduce_list(left: list | None, right: list | None) -> list:
    """Safely combine two lists, handling cases where either or both inputs might be None.

    Args:
        left (list | None): The first list to combine, or None.
        right (list | None): The second list to combine, or None.

    Returns:
        list: A new list containing all elements from both input lists.
               If an input is None, it's treated as an empty list.
    """
    if not left:
        left = []
    if not right:
        right = []
    return left + right

class DefaultState(TypedDict):
    foo: Annotated[list[int], add]

class CustomReducerState(TypedDict):
    foo: Annotated[list[int], reduce_list]

Compare the behavior: `DefaultState` crashes on `None`, while `CustomReducerState` handles `None` gracefully.

In [ ]:
def node_1(state):
    print("---Node 1---")
    return {"foo": [2]}

# Build graph
builder = StateGraph(DefaultState)
builder.add_node("node_1", node_1)

# Logic
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

try:
    print(graph.invoke({"foo" : None}))
except TypeError as e:
    print(f"TypeError occurred: {e}")

In [ ]:
# Build graph
builder = StateGraph(CustomReducerState)
builder.add_node("node_1", node_1)

# Logic
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

try:
    print(graph.invoke({"foo" : None}))
except TypeError as e:
    print(f"TypeError occurred: {e}")

## 7. The Built-in `add_messages` Reducer

In chat applications, `MessagesState` uses LangGraph's prebuilt **`add_messages`** reducer. It provides three critical capabilities:
1. **Append**: Appends new messages to the existing conversation list.
2. **Update**: If a message has an existing `id`, it overwrites that specific message in-place.
3. **Delete**: If a `RemoveMessage(id=...)` object is passed, it removes the corresponding message from state.

<div align="center">
  <img src="images/msg_reducer_reconciliation.png" alt="LangGraph add_messages Reducer Reconciliation Logic" width="100%" />
</div>

<br/>

<details>
<summary><b>🔍 View Raw Mermaid Diagram Syntax</b></summary>

```mermaid
flowchart LR
    subgraph Current [Current Message State]
        M1["AIMessage(id='1', 'Hello')"]
        M2["HumanMessage(id='2', 'Help me')"]
    end

    subgraph Operations [Incoming Operations]
        U2["HumanMessage(id='2', 'Updated: Help me now')"]
        M3["AIMessage(id='3', 'Sure!')"]
    end

    subgraph Reconciled [Result of add_messages]
        R1["AIMessage(id='1', 'Hello')"]
        R2["HumanMessage(id='2', 'Updated: Help me now')"]
        R3["AIMessage(id='3', 'Sure!')"]
    end

    Current & Operations --> Reconciled
```

</details>

In [ ]:
from typing import Annotated
from langgraph.graph import MessagesState
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

# Define a custom TypedDict that includes a list of messages with add_messages reducer
class CustomMessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    added_key_1: str
    added_key_2: str
    # etc

# Use MessagesState, which includes the messages key with add_messages reducer
class ExtendedMessagesState(MessagesState):
    # Add any keys needed beyond messages, which is pre-built 
    added_key_1: str
    added_key_2: str
    # etc

### Appending Messages with `add_messages`

In [ ]:
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage, HumanMessage

# Initial state
initial_messages = [AIMessage(content="Hello! How can I assist you?", name="Model"),
                    HumanMessage(content="I'm looking for information on marine biology.", name="Lance")
                   ]

# New message to add
new_message = AIMessage(content="Sure, I can help with that. What specifically are you interested in?", name="Model")

# Test
add_messages(initial_messages , new_message)

### Updating Messages In-Place by ID

In [ ]:
# Initial state
initial_messages = [AIMessage(content="Hello! How can I assist you?", name="Model", id="1"),
                    HumanMessage(content="I'm looking for information on marine biology.", name="Lance", id="2")
                   ]

# New message to add
new_message = HumanMessage(content="I'm looking for information on whales, specifically", name="Lance", id="2")

# Test
add_messages(initial_messages , new_message)

## 8. Pruning Context with `RemoveMessage`

To manage context window limits or remove stale tool calls, pass `RemoveMessage(id=...)` to `add_messages`. Any message matching that ID is purged from the state.

In [ ]:
from langchain_core.messages import RemoveMessage

# Message list
messages = [AIMessage("Hi.", name="Bot", id="1")]
messages.append(HumanMessage("Hi.", name="Lance", id="2"))
messages.append(AIMessage("So you said you were researching ocean mammals?", name="Bot", id="3"))
messages.append(HumanMessage("Yes, I know about whales. But what others should I learn about?", name="Lance", id="4"))

# Isolate messages to delete
delete_messages = [RemoveMessage(id=m.id) for m in messages[:-2]]
print(delete_messages)

In [ ]:
add_messages(messages , delete_messages)